# Notebook 02 - Data Cleaning

**Project:** What Does Poor Service Cost a Business? - Customer Operations Intelligence

**Objective:** Clean all four raw tables - handling missing values, duplicates, inconsistent labels, and invalid records. Every cleaning decision is documented with a reason. A data quality report is produced showing before and after for each table.

**Inputs:**
- `data/raw/transactions_raw.csv`
- `data/raw/customers_raw.csv`
- `data/raw/tickets_raw.csv`
- `data/raw/operations_raw.csv`

**Outputs:**
- `data/processed/transactions_clean.csv`
- `data/processed/customers_clean.csv`
- `data/processed/tickets_clean.csv`
- `data/processed/operations_clean.csv`
- `data/processed/data_quality_report.csv`

---

## 0. Setup

In [3]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)

# Quality report — we build this as we clean each table
quality_report = []

def log_quality(table, metric, before, after, action):
    """Record every cleaning decision for the quality report."""
    quality_report.append({
        'table': table,
        'metric': metric,
        'before': before,
        'after': after,
        'change': before - after if isinstance(before, (int, float)) else 'N/A',
        'action_taken': action
    })

print('Setup complete. Quality report initialised.')

Setup complete. Quality report initialised.


---
## 1. Clean Transactions Table

**Known issues to fix:**
- 243,007 missing Customer IDs (guest checkouts)
- 4,382 missing product descriptions
- 19,494 cancellation invoices (prefix 'C') with negative quantities
- Negative quantities from returns
- Zero and negative prices
- Negative revenue rows

In [4]:
print('Loading transactions...')
trans = pd.read_csv('data/raw/transactions_raw.csv')

print(f'Raw rows: {len(trans):,}')
print(f'Nulls:\n{trans.isnull().sum()}')
print(f'\nNegative quantities: {(trans["quantity"] < 0).sum():,}')
print(f'Zero or negative prices: {(trans["price"] <= 0).sum():,}')
print(f'Cancellation invoices: {trans["invoice"].astype(str).str.startswith("C").sum():,}')

Loading transactions...
Raw rows: 1,067,371
Nulls:
invoice              0
stock_code           0
description       4382
quantity             0
invoice_date         0
price                0
customer_id     243007
country              0
revenue              0
dtype: int64

Negative quantities: 22,950
Zero or negative prices: 6,207
Cancellation invoices: 19,494


In [5]:
rows_before = len(trans)

# --- Step 1: Remove cancellation invoices ---
# Reason: Invoices starting with 'C' are cancellations — they represent
# reversed transactions not actual sales. Including them distorts revenue analysis.
cancellations = trans['invoice'].astype(str).str.startswith('C').sum()
trans = trans[~trans['invoice'].astype(str).str.startswith('C')]
log_quality('transactions', 'cancellation invoices removed', cancellations, 0, 'Dropped — reversed transactions, not real sales')
print(f'After removing cancellations: {len(trans):,} rows')

# --- Step 2: Remove negative and zero quantities ---
# Reason: After removing cancellations, remaining negative quantities
# are likely data entry errors. Zero quantity = no transaction occurred.
neg_qty = (trans['quantity'] <= 0).sum()
trans = trans[trans['quantity'] > 0]
log_quality('transactions', 'zero/negative quantity rows', neg_qty, 0, 'Dropped — no valid transaction without positive quantity')
print(f'After removing bad quantities: {len(trans):,} rows')

# --- Step 3: Remove zero and negative prices ---
# Reason: Zero price items are likely samples or errors. Negative prices
# outside cancellations are data errors.
bad_price = (trans['price'] <= 0).sum()
trans = trans[trans['price'] > 0]
log_quality('transactions', 'zero/negative price rows', bad_price, 0, 'Dropped — no valid sale at zero or negative price')
print(f'After removing bad prices: {len(trans):,} rows')

# --- Step 4: Handle missing Customer IDs ---
# Reason: Guest checkouts have no customer_id. We keep these rows
# for revenue analysis but flag them as 'GUEST' so they are
# excluded from customer-level analysis in later notebooks.
missing_cust = trans['customer_id'].isnull().sum()
trans['customer_id'] = trans['customer_id'].fillna('GUEST')
trans['is_guest'] = trans['customer_id'] == 'GUEST'
log_quality('transactions', 'missing customer_id', missing_cust, 0, 'Filled with GUEST — retained for revenue, excluded from customer analysis')
print(f'Guest transactions flagged: {trans["is_guest"].sum():,}')

# --- Step 5: Handle missing descriptions ---
# Reason: Description is not critical for analysis — we fill with
# Unknown rather than dropping valid transactions.
missing_desc = trans['description'].isnull().sum()
trans['description'] = trans['description'].fillna('Unknown')
log_quality('transactions', 'missing description', missing_desc, 0, 'Filled with Unknown — description not critical for analysis')

# --- Step 6: Recalculate revenue on clean data ---
trans['revenue'] = trans['quantity'] * trans['price']

# --- Step 7: Ensure correct data types ---
trans['invoice_date'] = pd.to_datetime(trans['invoice_date'])
trans['customer_id'] = trans['customer_id'].astype(str)

rows_after = len(trans)
log_quality('transactions', 'total rows', rows_before, rows_after, f'Removed {rows_before - rows_after:,} invalid rows')

print(f'\nFinal clean rows: {len(trans):,}')
print(f'Remaining nulls:\n{trans.isnull().sum()}')

After removing cancellations: 1,047,877 rows
After removing bad quantities: 1,044,420 rows
After removing bad prices: 1,041,670 rows
Guest transactions flagged: 236,121

Final clean rows: 1,041,670
Remaining nulls:
invoice         0
stock_code      0
description     0
quantity        0
invoice_date    0
price           0
customer_id     0
country         0
revenue         0
is_guest        0
dtype: int64


In [6]:
trans.to_csv('data/processed/transactions_clean.csv', index=False)
print(f'transactions_clean.csv saved — {len(trans):,} rows')
print(trans.head(3))

transactions_clean.csv saved — 1,041,670 rows
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   

         invoice_date  price customer_id         country  revenue  is_guest  
0 2009-12-01 07:45:00   6.95     13085.0  United Kingdom     83.4     False  
1 2009-12-01 07:45:00   6.75     13085.0  United Kingdom     81.0     False  
2 2009-12-01 07:45:00   6.75     13085.0  United Kingdom     81.0     False  


---
## 2. Clean Customers Table

**Known issues to fix:**
- Inconsistent segment labels (Retail / retail / RETAIL)
- ~8% missing CLV values
- ~5% missing region values
- ~2% duplicate rows from system sync error

In [7]:
print('Loading customers...')
cust = pd.read_csv('data/raw/customers_raw.csv')

print(f'Raw rows: {len(cust):,}')
print(f'\nSegment values (before cleaning):')
print(cust['segment'].value_counts())
print(f'\nNulls:\n{cust.isnull().sum()}')
print(f'\nDuplicate rows: {cust.duplicated().sum():,}')

Loading customers...
Raw rows: 6,060

Segment values (before cleaning):
segment
Retail       3058
Corporate    1455
Wholesale     869
retail        303
corporate     198
RETAIL        177
Name: count, dtype: int64

Nulls:
customer_id              0
segment                  0
clv                    490
acquisition_channel      0
region                 300
acquisition_date         0
dtype: int64

Duplicate rows: 118


In [8]:
rows_before = len(cust)

# --- Step 1: Remove duplicate rows ---
# Reason: Duplicates came from a system sync error during data migration.
# Keeping duplicates would double-count customers in CLV and segment analysis.
dupes = cust.duplicated().sum()
cust = cust.drop_duplicates()
log_quality('customers', 'duplicate rows', dupes, 0, 'Dropped — system sync duplicates, would inflate customer counts')
print(f'After removing duplicates: {len(cust):,} rows')

# --- Step 2: Standardise segment labels ---
# Reason: The same three segments were entered inconsistently.
# All variations mapped to one clean label for accurate grouping.
segment_map = {
    'retail': 'Retail',
    'RETAIL': 'Retail',
    'Retail': 'Retail',
    'corporate': 'Corporate',
    'Corporate': 'Corporate',
    'Wholesale': 'Wholesale'
}
inconsistent_segments = cust['segment'].isin(['retail', 'RETAIL', 'corporate']).sum()
cust['segment'] = cust['segment'].map(segment_map)
log_quality('customers', 'inconsistent segment labels', inconsistent_segments, 0, 'Standardised to Retail / Corporate / Wholesale')
print(f'\nSegment values (after cleaning):')
print(cust['segment'].value_counts())

# --- Step 3: Handle missing CLV ---
# Reason: Missing CLV belongs to new customers with insufficient
# purchase history. We impute using the median CLV for their segment
# rather than dropping — losing customers from analysis is worse
# than using a conservative estimate.
missing_clv = cust['clv'].isnull().sum()
segment_medians = cust.groupby('segment')['clv'].median()
cust['clv_imputed'] = cust['clv'].isnull()  # flag which were imputed
cust['clv'] = cust.apply(
    lambda row: segment_medians[row['segment']] if pd.isnull(row['clv']) else row['clv'],
    axis=1
)
log_quality('customers', 'missing CLV values', missing_clv, 0, 'Imputed with segment median — new customers, conservative estimate')
print(f'\nCLV imputed for {missing_clv} customers')
print(f'Segment medians used: {segment_medians.round(2).to_dict()}')

# --- Step 4: Handle missing region ---
# Reason: Missing region is from a data migration gap.
# We label as Unknown rather than drop — these are still valid customers.
missing_region = cust['region'].isnull().sum()
cust['region'] = cust['region'].fillna('Unknown')
log_quality('customers', 'missing region', missing_region, 0, 'Filled with Unknown — data migration gap, customer still valid')

# --- Step 5: Ensure correct data types ---
cust['customer_id'] = cust['customer_id'].astype(str)
cust['clv'] = cust['clv'].round(2)
cust['acquisition_date'] = pd.to_datetime(cust['acquisition_date'])

rows_after = len(cust)
log_quality('customers', 'total rows', rows_before, rows_after, f'Removed {rows_before - rows_after} duplicate rows')

print(f'\nFinal clean rows: {len(cust):,}')
print(f'Remaining nulls:\n{cust.isnull().sum()}')

After removing duplicates: 5,942 rows

Segment values (after cleaning):
segment
Retail       3476
Corporate    1619
Wholesale     847
Name: count, dtype: int64

CLV imputed for 475 customers
Segment medians used: {'Corporate': 8067.01, 'Retail': 614.12, 'Wholesale': 2884.16}

Final clean rows: 5,942
Remaining nulls:
customer_id            0
segment                0
clv                    0
acquisition_channel    0
region                 0
acquisition_date       0
clv_imputed            0
dtype: int64


In [9]:
cust.to_csv('data/processed/customers_clean.csv', index=False)
print(f'customers_clean.csv saved — {len(cust):,} rows')
print(cust.head(3))

customers_clean.csv saved — 5,942 rows
  customer_id    segment       clv acquisition_channel        region  \
0       17976     Retail    614.12            Referral   Netherlands   
1       16608  Wholesale   1016.96              Online  Other Europe   
2       14596  Corporate  14379.43              Online        France   

  acquisition_date  clv_imputed  
0       2009-04-21         True  
1       2009-11-16        False  
2       2009-09-02        False  


---
## 3. Clean Tickets Table

**Known issues to fix:**
- 13 inconsistent issue category names for 6 actual categories
- ~18% missing CSAT scores (non-respondents)
- ~6% missing resolution times (abandoned tickets)
- SLA breach flag needs validation after cleaning

In [10]:
print('Loading tickets...')
tickets = pd.read_csv('data/raw/tickets_raw.csv')

print(f'Raw rows: {len(tickets):,}')
print(f'\nIssue categories (before cleaning):')
print(tickets['issue_category'].value_counts())
print(f'\nNulls:\n{tickets.isnull().sum()}')

Loading tickets...
Raw rows: 15,000

Issue categories (before cleaning):
issue_category
General Enquiry      2186
Delivery Delay       1524
Refund Request       1522
Billing Issue        1455
Login Issue          1264
Product Defect       1227
Account Access       1060
refund               1058
delivery delay        775
Billing               755
Defective Product     740
billing issue         740
Late Delivery         694
Name: count, dtype: int64

Nulls:
ticket_id             0
customer_id           0
location_id           0
ticket_date           0
issue_category        0
priority              0
resolution_days     900
sla_breached        900
csat_score         3464
status                0
dtype: int64


In [11]:
rows_before = len(tickets)

# --- Step 1: Standardise issue categories ---
# Reason: 13 category names represent only 6 actual issue types.
# Mixed casing and abbreviations were entered by different agents.
# Standardising enables accurate frequency and trend analysis by category.
category_map = {
    'Billing Issue':     'Billing',
    'billing issue':     'Billing',
    'Billing':           'Billing',
    'Delivery Delay':    'Delivery',
    'delivery delay':    'Delivery',
    'Late Delivery':     'Delivery',
    'Product Defect':    'Product Defect',
    'Defective Product': 'Product Defect',
    'Refund Request':    'Refund',
    'refund':            'Refund',
    'Account Access':    'Account Access',
    'Login Issue':       'Account Access',
    'General Enquiry':   'General Enquiry'
}
inconsistent_cats = tickets['issue_category'].isin(
    ['billing issue', 'Billing', 'delivery delay', 'Late Delivery',
     'Defective Product', 'refund', 'Login Issue']
).sum()
tickets['issue_category'] = tickets['issue_category'].map(category_map)
log_quality('tickets', 'inconsistent category names', inconsistent_cats, 0, 'Mapped 13 variants to 6 standard categories')
print('Issue categories (after cleaning):')
print(tickets['issue_category'].value_counts())

# --- Step 2: Handle missing resolution times ---
# Reason: ~6% of tickets were abandoned — no agent followed up.
# We keep these rows but flag them as abandoned.
# They are excluded from resolution time averages but counted
# in abandonment rate — which is itself a key performance metric.
missing_res = tickets['resolution_days'].isnull().sum()
tickets['is_abandoned'] = tickets['resolution_days'].isnull()
log_quality('tickets', 'missing resolution time (abandoned)', missing_res, 0, 'Flagged as abandoned — excluded from avg resolution, counted in abandonment rate')
print(f'\nAbandoned tickets flagged: {missing_res:,}')

# --- Step 3: Recalculate SLA breach on clean resolution data ---
# Reason: After cleaning, we recalculate the SLA breach flag
# to ensure it is consistent with the 2-day target.
# Abandoned tickets are marked as breached — no resolution = breach.
tickets['sla_breached'] = tickets.apply(
    lambda row: 1 if row['is_abandoned'] else (1 if row['resolution_days'] > 2 else 0),
    axis=1
)
print(f'SLA breached (recalculated): {tickets["sla_breached"].sum():,} tickets ({tickets["sla_breached"].mean()*100:.1f}%)')

# --- Step 4: Handle missing CSAT scores ---
# Reason: 18% of customers did not respond to the satisfaction survey.
# Non-response is NOT random — dissatisfied customers are less likely
# to respond. We do NOT impute CSAT. Instead we flag non-respondents
# and analyse response rate as a signal in itself.
missing_csat = tickets['csat_score'].isnull().sum()
tickets['csat_responded'] = tickets['csat_score'].notnull()
log_quality('tickets', 'missing CSAT (non-respondents)', missing_csat, missing_csat,
    'Retained as null — non-response is a signal, NOT imputed. Response rate analysed separately.')
print(f'\nCSAT response rate: {tickets["csat_responded"].mean()*100:.1f}%')

# --- Step 5: Ensure correct data types ---
tickets['customer_id'] = tickets['customer_id'].astype(str)
tickets['ticket_date'] = pd.to_datetime(tickets['ticket_date'])
tickets['resolution_days'] = pd.to_numeric(tickets['resolution_days'], errors='coerce')

rows_after = len(tickets)
log_quality('tickets', 'total rows', rows_before, rows_after, 'No rows dropped — all issues handled by flagging')

print(f'\nFinal clean rows: {len(tickets):,}')
print(f'Remaining nulls:\n{tickets.isnull().sum()}')

Issue categories (after cleaning):
issue_category
Delivery           2993
Billing            2950
Refund             2580
Account Access     2324
General Enquiry    2186
Product Defect     1967
Name: count, dtype: int64

Abandoned tickets flagged: 900
SLA breached (recalculated): 13,403 tickets (89.4%)

CSAT response rate: 76.9%

Final clean rows: 15,000
Remaining nulls:
ticket_id             0
customer_id           0
location_id           0
ticket_date           0
issue_category        0
priority              0
resolution_days     900
sla_breached          0
csat_score         3464
status                0
is_abandoned          0
csat_responded        0
dtype: int64


In [12]:
tickets.to_csv('data/processed/tickets_clean.csv', index=False)
print(f'tickets_clean.csv saved — {len(tickets):,} rows')
print(tickets.head(3))

tickets_clean.csv saved — 15,000 rows
   ticket_id customer_id location_id ticket_date issue_category priority  \
0  TKT-00001       13641      LOC_02  2011-07-12       Delivery      Low   
1  TKT-00002       15780      LOC_03  2011-07-04         Refund      Low   
2  TKT-00003       15481      LOC_01  2011-05-16         Refund   Medium   

   resolution_days  sla_breached  csat_score     status  is_abandoned  \
0              6.2             1         2.0   Resolved         False   
1              8.2             1         3.0       Open         False   
2              3.7             1         4.0  Escalated         False   

   csat_responded  
0            True  
1            True  
2            True  


---
## 4. Clean Operations Table

**Known issues to fix:**
- Inconsistent location name: 'Location_03' should be 'LOC_03'
- Missing SLA achievement rate for 2 location-quarter combinations
- Missing staff count for LOC_08 across all quarters

In [13]:
print('Loading operations...')
ops = pd.read_csv('data/raw/operations_raw.csv')

print(f'Raw rows: {len(ops):,}')
print(f'\nLocation IDs (before cleaning):')
print(ops['location_id'].value_counts())
print(f'\nNulls:\n{ops.isnull().sum()}')

Loading operations...
Raw rows: 64

Location IDs (before cleaning):
location_id
LOC_01         8
LOC_02         8
LOC_04         8
LOC_05         8
LOC_06         8
LOC_07         8
LOC_08         8
LOC_03         7
Location_03    1
Name: count, dtype: int64

Nulls:
location_id              0
quarter                  0
region                   0
staff_count              8
sla_target_days          0
sla_achievement_rate     2
inventory_level          0
ticket_volume_handled    0
dtype: int64


In [14]:
rows_before = len(ops)

# --- Step 1: Fix inconsistent location name ---
# Reason: 'Location_03' is the same location as 'LOC_03' —
# a data entry error in Q2 2010. Corrected to match the standard
# format used across all other tables.
bad_name_count = (ops['location_id'] == 'Location_03').sum()
ops['location_id'] = ops['location_id'].replace('Location_03', 'LOC_03')
log_quality('operations', 'inconsistent location name', bad_name_count, 0, "'Location_03' corrected to 'LOC_03' — data entry error in Q2 2010")
print(f'Location names fixed: {bad_name_count} row(s)')
print('Location IDs (after cleaning):')
print(ops['location_id'].value_counts())

# --- Step 2: Handle missing SLA achievement rate ---
# Reason: Two location-quarter combinations were not reported.
# We impute using that location's average SLA achievement across
# other quarters — a conservative and defensible estimate.
missing_sla = ops['sla_achievement_rate'].isnull().sum()
ops['sla_imputed'] = ops['sla_achievement_rate'].isnull()
location_avg_sla = ops.groupby('location_id')['sla_achievement_rate'].mean()
ops['sla_achievement_rate'] = ops.apply(
    lambda row: location_avg_sla[row['location_id']]
    if pd.isnull(row['sla_achievement_rate']) else row['sla_achievement_rate'],
    axis=1
).round(2)
log_quality('operations', 'missing SLA achievement rate', missing_sla, 0, 'Imputed with location average — unreported quarters, conservative estimate')
print(f'\nSLA achievement imputed for {missing_sla} row(s)')

# --- Step 3: Handle missing staff count ---
# Reason: LOC_08 staff count was never recorded — data gap from
# onboarding. We impute using the median staff count across all
# locations and flag these rows clearly.
missing_staff = ops['staff_count'].isnull().sum()
ops['staff_imputed'] = ops['staff_count'].isnull()
median_staff = ops['staff_count'].median()
ops['staff_count'] = ops['staff_count'].fillna(median_staff)
log_quality('operations', 'missing staff count', missing_staff, 0, f'Imputed with median ({median_staff}) — LOC_08 onboarding gap')
print(f'Staff count imputed for {missing_staff} row(s) using median: {median_staff}')

# --- Step 4: Add SLA performance classification ---
# Reason: A binary flag makes it easier to filter and compare
# underperforming vs compliant locations in analysis.
ops['sla_status'] = ops['sla_achievement_rate'].apply(
    lambda x: 'Critical' if x < 0.50 else ('At Risk' if x < 0.70 else 'Compliant')
)
print(f'\nSLA status breakdown:')
print(ops['sla_status'].value_counts())

rows_after = len(ops)
log_quality('operations', 'total rows', rows_before, rows_after, 'No rows dropped — all issues resolved by correction or imputation')

print(f'\nFinal clean rows: {len(ops):,}')
print(f'Remaining nulls:\n{ops.isnull().sum()}')

Location names fixed: 1 row(s)
Location IDs (after cleaning):
location_id
LOC_01    8
LOC_02    8
LOC_03    8
LOC_04    8
LOC_05    8
LOC_06    8
LOC_07    8
LOC_08    8
Name: count, dtype: int64

SLA achievement imputed for 2 row(s)
Staff count imputed for 8 row(s) using median: 12.0

SLA status breakdown:
sla_status
Compliant    48
At Risk      11
Critical      5
Name: count, dtype: int64

Final clean rows: 64
Remaining nulls:
location_id              0
quarter                  0
region                   0
staff_count              0
sla_target_days          0
sla_achievement_rate     0
inventory_level          0
ticket_volume_handled    0
sla_imputed              0
staff_imputed            0
sla_status               0
dtype: int64


In [15]:
ops.to_csv('data/processed/operations_clean.csv', index=False)
print(f'operations_clean.csv saved — {len(ops):,} rows')
print(ops)

operations_clean.csv saved — 64 rows
   location_id  quarter       region  staff_count  sla_target_days  \
0       LOC_01  2010-Q1           UK          9.0              2.0   
1       LOC_02  2010-Q1           UK         12.0              2.0   
2       LOC_03  2010-Q1      Germany          3.0              2.0   
3       LOC_04  2010-Q1       France          8.0              2.0   
4       LOC_05  2010-Q1  Netherlands          8.0              2.0   
..         ...      ...          ...          ...              ...   
59      LOC_04  2011-Q4       France         12.0              2.0   
60      LOC_05  2011-Q4  Netherlands         10.0              2.0   
61      LOC_06  2011-Q4         EIRE         10.0              2.0   
62      LOC_07  2011-Q4      Germany         10.0              2.0   
63      LOC_08  2011-Q4       France         12.0              2.0   

    sla_achievement_rate  inventory_level  ticket_volume_handled  sla_imputed  \
0                   0.82             0.92

---
## 5. Data Quality Report

A summary of every cleaning decision made across all four tables. This is a standard deliverable in professional BI workflows — it documents data lineage and justifies analytical choices to stakeholders.

In [16]:
quality_df = pd.DataFrame(quality_report)

print('=' * 70)
print('  DATA QUALITY REPORT')
print('=' * 70)
print(quality_df.to_string(index=False))

quality_df.to_csv('data/processed/data_quality_report.csv', index=False)
print('\ndata_quality_report.csv saved.')

  DATA QUALITY REPORT
       table                              metric  before   after change                                                                                 action_taken
transactions       cancellation invoices removed   19494       0    N/A                                              Dropped — reversed transactions, not real sales
transactions         zero/negative quantity rows    3457       0    N/A                                     Dropped — no valid transaction without positive quantity
transactions            zero/negative price rows    2750       0    N/A                                            Dropped — no valid sale at zero or negative price
transactions                 missing customer_id  236121       0    N/A                    Filled with GUEST — retained for revenue, excluded from customer analysis
transactions                 missing description       0       0    N/A                                  Filled with Unknown — description not critical f

---
## 6. Final Summary — Cleaning Complete

In [17]:
print('=' * 55)
print('  DATA CLEANING COMPLETE')
print('=' * 55)
print()

files = [
    ('transactions_clean.csv', trans),
    ('customers_clean.csv', cust),
    ('tickets_clean.csv', tickets),
    ('operations_clean.csv', ops)
]

for fname, df in files:
    nulls = df.isnull().sum().sum()
    print(f'  data/processed/{fname}')
    print(f'    Rows: {len(df):,}  |  Remaining nulls: {nulls}')
    print()

print(f'  Cleaning decisions documented: {len(quality_report)}')
print()
print('All four clean tables saved to data/processed/')
print('Next: Run Notebook 03 — Exploratory Analysis')

  DATA CLEANING COMPLETE

  data/processed/transactions_clean.csv
    Rows: 1,041,670  |  Remaining nulls: 0

  data/processed/customers_clean.csv
    Rows: 5,942  |  Remaining nulls: 0

  data/processed/tickets_clean.csv
    Rows: 15,000  |  Remaining nulls: 4364

  data/processed/operations_clean.csv
    Rows: 64  |  Remaining nulls: 0

  Cleaning decisions documented: 19

All four clean tables saved to data/processed/
Next: Run Notebook 03 — Exploratory Analysis
